# Fertilizer RAS — real FAOSTAT data (Nitrogen)

End-to-end run on real data for **one nutrient** (N), with a Russia/Belarus/Ukraine production shock.
All heavy lifting lives in `src/`; this notebook just calls into it.

Before running this notebook, download the FAOSTAT bulk extracts as described in `data/README.md`.

In [ ]:
import sys, pathlib

sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd

from src.preprocessing import load_nutrient, apply_shock_reported
from src.model import FertilizerRAS
from src.postprocessing import (
    build_comparison, most_affected, least_affected,
    global_summary, sanity_checks, save_result,
)
from src.utils import plot_comparison_dashboard

pd.set_option('display.float_format', '{:,.0f}'.format)

## 1. Configuration

In [ ]:
NUTRIENT = 'N'           # one of 'N', 'P', 'K'
YEARS = list(range(2014, 2019))
MIN_THRESHOLD = 1_000    # tonnes; drop countries with tiny P and C

INPUTS_CSV = '../data/Inputs_FertilizersNutrient_E_All_Data/Inputs_FertilizersNutrient_E_All_Data_NOFLAG.csv'
TRADE_CSV  = '../data/Fertilizers_DetailedTradeMatrix_E_All_Data/Fertilizers_DetailedTradeMatrix_E_All_Data_NOFLAG.csv'

SHOCK = {
    'Russian Federation': 0.40,   # 60% cut
    'Belarus':            0.30,   # 70% cut
    'Ukraine':            0.20,   # 80% cut
}

## 2. Load FAOSTAT data

In [ ]:
data = load_nutrient(INPUTS_CSV, TRADE_CSV, nutrient=NUTRIENT, years=YEARS, min_threshold=MIN_THRESHOLD)

P, C, T0 = data['P'], data['C'], data['T0']
print(f'Countries: {len(data["countries"])}')
print(f'Total production: {P.sum():,.0f} t')
print(f'Total demand:     {C.sum():,.0f} t')
print(f'Total trade (T0): {T0.values.sum():,.0f} t')

display(P.nlargest(10).to_frame('Production (t)'))
display(C.nlargest(10).to_frame('Demand (t)'))

## 3. Apply shock and run baseline + shocked

In [ ]:
P_shocked, report = apply_shock_reported(P, SHOCK)
display(report.as_dataframe())
if report.unmatched:
    print('Unmatched country names in SHOCK dict:', report.unmatched)

baseline = FertilizerRAS(P,         C, T0).run(verbose=True)
shocked  = FertilizerRAS(P_shocked, C, T0).run(verbose=True)

display(sanity_checks(baseline))
display(sanity_checks(shocked))

## 4. Compare and report

In [ ]:
comparison = build_comparison(baseline, shocked)

print('Global summary:')
display(global_summary(baseline, shocked))

print('\nMost affected:')
display(most_affected(comparison, n=15))

print('\nLeast affected / improved:')
display(least_affected(comparison, n=10))

## 5. Plots

In [ ]:
fig = plot_comparison_dashboard(baseline, shocked, comparison, nutrient_name=f'Nutrient {NUTRIENT}')
fig.show()

## 6. Save results to `results/`

In [ ]:
save_result(baseline, '../results', tag=f'{NUTRIENT}_baseline')
save_result(shocked,  '../results', tag=f'{NUTRIENT}_shocked')